# Analysis Template: Michaelis–Menten
Testing for units support on ADK data, 12/19/25

Updated 9/17/25, Nicholas Freitas

Data structures are in htbam_db_api. 
We need to make sure that transforms, fitting, masking work with units

Refactoring for Unit Handling::
- Add units array to data structure (x)
- Add units to transforms (x)
- Add units to fitting  (x)
- Add units to masking (should work out of the box?) (x?)


# Outstanding work:

Checked over with Jacob.
Data looks good (our fitting here is much better, which explains the discrepancy).
May need to add auto window-finding in the fitting function.

Current mystery: my kcat stdev's are much higher than Jacob's. Can't find a reason why right now... I'll have to dig into the code and look at his data + notebook. I'm asking for his notebook and conda env file so I can replicate his results.

Next I'll need to add the new z-score filtering, background subtraction, PDF output, and new CSV output:

CSV with average initial rates per-chamber, with stdevs and number of replicates after filtering. (list positions)

for initial rate finding:
- potentially cut off beginning
- Find best possible window for linear fit
- Currently we define a large window and get a small subwindow with low R2, with minimum and maximum # of points. But, this can lead to situations where we get shittier fits with larger R2 because we're using more points (like with Jacob's data).
- Instead, what if we fit like every 3 points, and see at what point the slope starts to change (the curve starts flattening out?). Then we collect all the points within the best slopes and fit a final time, with R2.


Transforms:
- Device transforms and sample transforms may be borked, have to test later. Chambers work great.

In [2]:
#enables autoreloding of modules
%load_ext autoreload
%autoreload 2

from htbam_db_api.htbam_db_api import LocalHtbamDBAPI
from htbam_analysis.analysis.experiment import HTBAMExperiment
from htbam_analysis.analysis import plot
from htbam_db_api.units import units

#enable inline plotting of matplotlib figures
%matplotlib inline

#set the figure format to SVG
%config InlineBackend.figure_format = 'svg'

## 1. Connect DB Api

In [3]:
### PARAMETERS:
EGFP_SLOPE = 91900.03 / 9 * (units.RFU / units.nM) # Ruensern correction factor (RCF)

#EGFP_SLOPE_CONC_UNITS = 'nM' #RFU/nM

root = '../example_data/9_17_25/mm_adk_data/'
db_conn = LocalHtbamDBAPI(
    standard_curve_data_path= root + 'd1_NADPH_500ms_StandardSeries_Analysis.csv',
    standard_name="NADPH", 
    standard_substrate="NADPH", 
    standard_units=units.uM,
    kinetic_data_path= root+ 'd1_500ms_TitrationSeries_Analysis.csv',
    kinetic_name="ADP", 
    kinetic_substrate="ADP", 
    kinetic_units=units.uM,
    time_units=units.s)

htbam_experiment = HTBAMExperiment(db_conn)


Connected to database.
Experiment found with the following runs:
['NADPH', 'ADP', 'button_quant']


/home/nfreitas/workspace/htbam_refactor_12_10_25/htbam_db_api/src/htbam_db_api/htbam_db_api.py:100: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  dep_var=np.array(button_quant_sum),  # (n_chambers, 1)


## 2. Enzyme Quant

In [4]:
from htbam_analysis.analysis.transform import transform_data

button_concentrations = transform_data(
    data_objs = [htbam_experiment.get_run('button_quant')],              # e.g. a Data2D or Data3D or Data4D instance
    expr=f'(a_luminance / slope)',             # e.g. "(luminance - intercept) / slope"
    expression_vars={'slope': EGFP_SLOPE},
    output_name='concentration'       # name of the new field, e.g. "concentration"
)

htbam_experiment.set_run('enzyme_concentrations', button_concentrations)     # save the fit results

Existing run data not found. Fetching from database.
Transforming 1 objects of type [<class 'htbam_db_api.data.Data2D'>] with shapes [(1792, 1)].
Evaluating expression: (a_luminance / slope)


In [5]:
plot.plot_enzyme_concentration_chip(htbam_experiment, analysis_name='enzyme_concentrations')

## 2. Product Standards

In [6]:
from htbam_analysis.analysis.fit import fit_luminance_vs_concentration

# For demonstration purposes, I'm doing this in 3 lines
# we can either keep it "exposed" like this (might be useful?)
# Or provide a simple wrapper fx that looks like:
# htbam_experiment.fit_standard_curve(experiment_name = 'standard_5-FAM', analysis_name='htbam_experiment')

standard_experiment_data = htbam_experiment.get_run('NADPH')       # retrieve our raw data
standard_fits = fit_luminance_vs_concentration(standard_experiment_data)    # perform a fit
htbam_experiment.set_run('NADPH_standard', standard_fits)              # save the fit results

Existing run data not found. Fetching from database.
Fit slopes for 1792 wells.
Elapsed 1.935 seconds.


In [7]:
## Here, I'm showing how to create a custom mask to ignore bad std curve timepoints

# from htbam_analysis.analysis.fit import fit_luminance_vs_concentration
# from htbam_analysis.analysis.filter import make_custom_mask
# import numpy as np

# standard_experiment_data = htbam_experiment.get_run('NADPH')       # retrieve our raw data
# standard_experiment_data.dep_var.shape # (8,1, 1792, 1)

# # let's mask out the final concentration
# mask = np.ones(standard_experiment_data.dep_var.shape) * True
# mask[-1, :,:,:] = False
# mask = mask.astype(bool)
# custom_mask = make_custom_mask(standard_experiment_data, mask, info='final_timepoint_mask')
# htbam_experiment.set_run('final_timepoint_mask', custom_mask)

# htbam_experiment.apply_mask(run_name='NADPH', 
#                             dep_variables = ['luminance'], 
#                             save_as = 'NADPH_masked',
#                             mask_names = ['final_timepoint_mask'])

# standard_experiment_data = htbam_experiment.get_run('NADPH_masked')       # retrieve our raw data
# standard_fits = fit_luminance_vs_concentration(standard_experiment_data)    # perform a fit
# htbam_experiment.set_run('NADPH_standard', standard_fits)              # save the fit results


In [8]:
plot.plot_standard_curve_chip(htbam_experiment, 'NADPH_standard', 'NADPH')

## 3. Fit Initial Rates

In [9]:
# Calculate product concentrations from RFU data:
product_concentrations = transform_data(
    data_objs = [htbam_experiment.get_run('ADP'), htbam_experiment.get_run('NADPH_standard')],
    expr=f'(a.luminance - b.intercept) / b.slope',             # e.g. "(luminance - intercept) / slope"
    output_name='concentration'       # name of the new field, e.g. "concentration"
)
 
htbam_experiment.set_run('kinetics_ADP_conc', product_concentrations)

Existing run data not found. Fetching from database.
Transforming 2 objects of type [<class 'htbam_db_api.data.Data4D'>, <class 'htbam_db_api.data.Data2D'>] with shapes [(9, 20, 1792, 1), (1792, 3)].
Evaluating expression: (a.luminance - b.intercept) / b.slope


In [10]:
from htbam_analysis.analysis.fit import fit_concentration_vs_time

kinetics_concentrations = htbam_experiment.get_run('kinetics_ADP_conc')       # retrieve our raw data
kinetics_fits = fit_concentration_vs_time(kinetics_concentrations, start_timepoint = 1, end_timepoint=6)      # perform a fit, skipping the first timepoint
htbam_experiment.set_run('kinetics_ADP_conc_fits', kinetics_fits)                # save the fit results

Fit slopes for 1792 wells at 9 concentrations.
Elapsed 0.202 seconds.


In [11]:
## Example for overriding the fit windows per concentration:
# kinetics_fits = fit_concentration_vs_time(kinetics_concentrations,
#                                            fit_windows_per_concentration=
#                                            {0.5: (1, 20),
#                                             1.0: (1, 6),
#                                             2.0: (1, 6),
#                                             4.0: (1, 6),
#                                             8.0: (1, 6),
#                                             16.0: (1, 6),
#                                             32.0: (1, 6),
#                                             64.0: (1, 6),
#                                             128.0: (1, 6),
#                                             256.0: (1, 20)})

In [12]:
plot.plot_initial_rates_chip(htbam_experiment, analysis_name='kinetics_ADP_conc_fits', experiment_name='kinetics_ADP_conc')#, remove_0_point=True) # plot the initial rates

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/pint/facets/numpy/numpy_func.py:856: RuntimeWarning:

Mean of empty slice



## 3.5 Background Rate Subtraction

In [13]:
from htbam_analysis.analysis.filter import filter_by_sample_id

# Here, we create a mask of the data to only have the buffer wells. 
buffer_mask =  filter_by_sample_id(kinetics_fits, 
                                    sample_ids=['buffer'])
htbam_experiment.set_run('buffer_mask', buffer_mask)

# Apply the masks to the fits and save the results
htbam_experiment.apply_mask(run_name='kinetics_ADP_conc_fits', 
                            dep_variables = ['slope', 'intercept', 'r_squared'], 
                            save_as = 'buffer_wells',
                            mask_names = ['buffer_mask'])

# Now, we'll do processing on the buffer wells:
buffer_wells = htbam_experiment.get_run('buffer_wells')

# In each well, we'll store the 25th percentile of all buffer slopes.
buffer_wells_lower_quartile = transform_data(
    data_objs = [buffer_wells],              # e.g. a Data2D or Data3D or Data4D instance
    expr=f'np.nanpercentile(a.device.slope, 25, axis=1)',             # e.g. "(luminance - intercept) / slope"
    output_name='slope_lower_quartile'       # name of the new field, e.g. "slope"
)

htbam_experiment.set_run('buffer_wells_lower_quartile', buffer_wells_lower_quartile)     # save the fit results

# Now, we'll transform the data to subtract the background rates, per-concentration:
corrected_initial_rates = transform_data(
    data_objs = [htbam_experiment.get_run('kinetics_ADP_conc_fits'), htbam_experiment.get_run('buffer_wells_lower_quartile')],              # e.g. a Data2D or Data3D or Data4D instance
    expr=f'(a.chamber.slope - b.chamber.slope_lower_quartile)',             # e.g. "(luminance - intercept) / slope"
    output_name='slope',       # name of the new field, e.g. "slope"
    keep_existing = True   # keep the existing fields (slope, intercept, r_squared)
)

htbam_experiment.set_run('kinetics_ADP_conc_fits_bgsub', corrected_initial_rates)     # save the fit results


Transforming 1 objects of type [<class 'htbam_db_api.data.Data3D'>] with shapes [(9, 1792, 3)].
Evaluating expression: np.nanpercentile(a.device.slope, 25, axis=1)
Transforming 2 objects of type [<class 'htbam_db_api.data.Data3D'>, <class 'htbam_db_api.data.Data3D'>] with shapes [(9, 1792, 3), (9, 1792, 1)].
Evaluating expression: (a.chamber.slope - b.chamber.slope_lower_quartile)


## 4. Filter initial rates

In [14]:
# htbam_experiment.filter_initial_rates('kinetic_0',
#                                       'standard_0',
#                                       standard_curve_r2_cutoff = 0.98,
#                                       expression_threshold = 0.5,
#                                       initial_rate_R2_threshold = 0.8, 
#                                       positive_initial_slope_filter = True,
#                                       background_subtraction = True,)

In [15]:
# These are the "Masks" we use to select which data we want.

# From Duncan's paper:
#standard_curve_r2_cutoff = 0.98,
#expression_threshold = 0.5,
#initial_rate_R2_threshold = 0.8, 
#positive_initial_slope_filter = True,
#background_subtraction = True,

from htbam_analysis.analysis.filter import filter_expression_cutoff, filter_initial_rates_positive_cutoff, filter_initial_rates_r2_cutoff, filter_standard_curve_r2_cutoff

kinetics_fits = htbam_experiment.get_run('kinetics_ADP_conc_fits_bgsub')       # retrieve our raw data

# Initial Rates:
initial_rates_r2_mask =         filter_initial_rates_r2_cutoff(kinetics_fits, r2_cutoff=0.8) # 0.9 R2
initial_rates_positive_mask =   filter_initial_rates_positive_cutoff(kinetics_fits)          # positive slope

# Standard Curve:
standard_curve_r2_mask =        filter_standard_curve_r2_cutoff(standard_fits, kinetics_fits, r2_cutoff=0.98) # 0.9 R2

# Expression:
expression_mask =               filter_expression_cutoff(button_concentrations, kinetics_fits, expression_cutoff=0.5) # 1 nM

# Save the masks to the experiment
htbam_experiment.set_run('initial_rates_r2_mask', initial_rates_r2_mask)
htbam_experiment.set_run('initial_rates_positive_mask', initial_rates_positive_mask)
htbam_experiment.set_run('standard_curve_r2_mask', standard_curve_r2_mask)
htbam_experiment.set_run('expression_mask', expression_mask)



In [16]:
# Plot the masks (Do one at a time)
#plot.plot_mask_chip(htbam_experiment, mask_name='initial_rates_r2_mask')
#plot.plot_mask_chip(htbam_experiment, mask_name='initial_rates_positive_mask')
#plot.plot_mask_chip(htbam_experiment, mask_name='standard_curve_r2_mask')
plot.plot_mask_chip(htbam_experiment, mask_name='expression_mask')

In [17]:
# Apply the masks to the fits and save the results
htbam_experiment.apply_mask(run_name='kinetics_ADP_conc_fits_bgsub', 
                            dep_variables = ['slope', 'intercept'], 
                            save_as = 'kinetics_ADP_conc_fits_masked',
                            mask_names = ['initial_rates_r2_mask', 'initial_rates_positive_mask', 'expression_mask'])


In [18]:
# Now, we'll delete any wells that have fewer than 5 initial rates:
from htbam_analysis.analysis.filter import filter_number_concentrations

kinetics_fits = htbam_experiment.get_run('kinetics_ADP_conc_fits_masked')       # retrieve our raw data
number_initial_rates_mask = filter_number_concentrations(kinetics_fits, min_concentrations=5, var_to_check='slope')
htbam_experiment.set_run('number_initial_rates_mask', number_initial_rates_mask)
htbam_experiment.apply_mask(run_name='kinetics_ADP_conc_fits_masked', 
                            dep_variables = ['slope', 'intercept', 'r_squared'],
                            mask_names=['number_initial_rates_mask'],
                            save_as = 'kinetics_ADP_conc_fits_masked') # Overwrite the previous masked data

In [19]:
# Note - because of background subtraction, the lines may fall below the points!!!

plot.plot_initial_rates_chip(experiment=htbam_experiment, analysis_name='kinetics_ADP_conc_fits_masked', experiment_name='kinetics_ADP_conc')#, remove_0_point=True) # plot the initial rates

In [20]:
plot.plot_initial_rates_vs_concentration_chip(experiment=htbam_experiment, analysis_name='kinetics_ADP_conc_fits_masked') # plot the initial rates

## 5. Fit MM Constant:

In [21]:
# I've written MM here, but it will be IC50
from htbam_analysis.analysis.fit import fit_initial_rates_vs_concentration_with_function, mm_model, inhibition_model

kinetics_fits_masked = htbam_experiment.get_run('kinetics_ADP_conc_fits_masked')       # retrieve our raw data
MM_fits, MM_pred_data = fit_initial_rates_vs_concentration_with_function(data = kinetics_fits_masked,
                                model_func = mm_model)
htbam_experiment.set_run('kinetics_ADP_MM_fits', MM_fits)                # save the fit results
htbam_experiment.set_run('kinetics_ADP_MM_pred_data', MM_pred_data)    # save the predicted data

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/scipy/optimize/_minpack_py.py:881: OptimizeWarning:

Covariance of the parameters could not be estimated



Successfully fit nonlinear model for 1607 wells.
Elapsed 0.72 seconds.


In [22]:
from htbam_analysis.analysis.filter import filter_r2_cutoff, filter_number_replicates
# Filter out crappy R2 values: 
MM_fits = htbam_experiment.get_run('kinetics_ADP_MM_fits')

MM_fits_mask = filter_r2_cutoff(MM_fits, r2_cutoff=0.0)  #  No R2 filter for now...
MM_fits_replicate_mask = filter_number_replicates(MM_fits, min_replicates=5, var_to_check='K_m')

htbam_experiment.set_run('MM_R2_mask', MM_fits_mask)  # save the fit results
htbam_experiment.set_run('MM_replicate_mask', MM_fits_replicate_mask)

plot.plot_mask_chip(experiment=htbam_experiment, mask_name='MM_replicate_mask')

In [23]:
# Apply the mask to the fits and save the results:
htbam_experiment.apply_mask(run_name='kinetics_ADP_MM_fits', 
                            dep_variables = ['v_max', 'K_m', 'r_squared'], 
                            save_as = 'kinetics_ADP_MM_fits_masked',
                            mask_names = ['MM_R2_mask', 'MM_replicate_mask'])

In [24]:
plot.plot_MM_chip(experiment=htbam_experiment, analysis_name='kinetics_ADP_conc_fits_masked', 
                                model_fit_name='kinetics_ADP_MM_fits_masked',
                                model_pred_data_name='kinetics_ADP_MM_pred_data',
                                )#x_log=True) # plot the initial rates

# 6. Divide by $[E]$ to get $k_{cat}$, $V_0/[E]$

In [25]:
# This is the same data we made above, we're just going to 
# divide everything by enzyme concentration basically

# enzyme concentration:
enzyme_concentrations = htbam_experiment.get_run("enzyme_concentrations")

# masked slopes vs conc:
masked_slopes_vs_conc = htbam_experiment.get_run("kinetics_ADP_conc_fits_masked")
# MM masked fits:
masked_fits = htbam_experiment.get_run("kinetics_ADP_MM_fits_masked")
# pred data from MM fits:
pred_data = htbam_experiment.get_run("kinetics_ADP_MM_pred_data")

# Get V_0/[E]:
masked_V0_div_E_vs_conc = transform_data(
    data_objs = [masked_slopes_vs_conc, enzyme_concentrations],
    expr=f'a_slope / b_concentration',             # e.g. "(luminance - intercept) / slope"
    output_name='V0_div_E'       # name of the new field, e.g. "concentration"
)

# Get V_max/[E] (kcat)
masked_fits_with_kcat = transform_data(
    data_objs = [masked_fits, enzyme_concentrations],
    expr=f'a_v_max / b_concentration',             # e.g. "(luminance - intercept) / slope"
    output_name='kcat',       # name of the new field, e.g. "concentration",
    keep_existing = True
)

# # Do the same for pred data:
pred_data_div_E = transform_data(
    data_objs = [pred_data, enzyme_concentrations],
    expr=f'a_y_pred / b_concentration',             # e.g. "(luminance - intercept) / slope"
    output_name='pred_V0_div_E',       # name of the new field, e.g. "concentration",
)

# Add to experiment
htbam_experiment.set_run('masked_V0_div_E_vs_conc', masked_V0_div_E_vs_conc)
htbam_experiment.set_run('masked_fits_with_kcat', masked_fits_with_kcat)
htbam_experiment.set_run('pred_data_div_E', pred_data_div_E)

Transforming 2 objects of type [<class 'htbam_db_api.data.Data3D'>, <class 'htbam_db_api.data.Data2D'>] with shapes [(9, 1792, 3), (1792, 1)].
Evaluating expression: a_slope / b_concentration
Transforming 2 objects of type [<class 'htbam_db_api.data.Data2D'>, <class 'htbam_db_api.data.Data2D'>] with shapes [(1792, 3), (1792, 1)].
Evaluating expression: a_v_max / b_concentration
Transforming 2 objects of type [<class 'htbam_db_api.data.Data3D'>, <class 'htbam_db_api.data.Data2D'>] with shapes [(9, 1792, 1), (1792, 1)].
Evaluating expression: a_y_pred / b_concentration


## Remove Outliers

In [ ]:
# Old reference z-fitting code here:
#https://github.com/pinneylab/htbam_analysis/blob/b0b1f1737978fd6c258ab3c7b9e2452de18bc2e7/src/htbam_analysis/analysis/experiment.py


# NaN all chamber values where K_m is 3 standard deviations away from the mean
z_score_threshold = 1.5

# # First, we manually remove some crazy values
# MM_outlier_mask_manual = transform_data(
#     data_objs = [htbam_experiment.get_run('masked_fits_with_kcat')],
#     expr=f'a.chamber.K_m.magnitude < 1000',
#     output_name='mask'
# )

# htbam_experiment.set_run('MM_outlier_mask_manual', MM_outlier_mask_manual)
# #plot.plot_mask_chip(htbam_experiment, mask_name='MM_outlier_mask_manual')

# # Apply the mask to the fits and save the results:
# htbam_experiment.apply_mask(run_name='masked_fits_with_kcat',
#                             dep_variables = ['v_max', 'K_m', 'r_squared', 'kcat'], 
#                             save_as = 'masked_fits_with_kcat_filtered',
#                             mask_names = ['MM_outlier_mask_manual'])

# First, we make a boolean mask of all chambers that are within 3 standard deviations of the mean K_m across the entire device.
MM_outlier_mask_KM = transform_data(
    data_objs = [htbam_experiment.get_run('masked_fits_with_kcat')], # or use masked_fits_with_kcat_filtered
    expr=f'~(np.abs(a.chamber.K_m - np.nanmean(a.sample.K_m)) > z_score_threshold * np.nanstd(a.sample.K_m))',
    expression_vars={'z_score_threshold': z_score_threshold},
    output_name='mask'
)

MM_outlier_mask_kcat = transform_data(
    data_objs = [htbam_experiment.get_run('masked_fits_with_kcat')], # or use masked_fits_with_kcat_filtered
    expr=f'~(np.abs(a.chamber.kcat - np.nanmean(a.sample.kcat)) > z_score_threshold * np.nanstd(a.sample.kcat))',
    expression_vars={'z_score_threshold': z_score_threshold},
    output_name='mask'
)

MM_outlier_mask_expression = transform_data(
    data_objs = [htbam_experiment.get_run('enzyme_concentrations')], # or use masked_fits_with_kcat_filtered
    expr=f'~(np.abs(a.chamber.concentration - np.nanmean(a.sample.concentration)) > z_score_threshold * np.nanstd(a.sample.concentration))',
    expression_vars={'z_score_threshold': z_score_threshold},
    output_name='mask'
)

# And them all together:
MM_outlier_mask = transform_data(
    data_objs = [MM_outlier_mask_KM, MM_outlier_mask_kcat, MM_outlier_mask_expression],
    expr=f'a.mask.magnitude & b.mask.magnitude & c.mask.magnitude',
    output_name='mask'
)

htbam_experiment.set_run('MM_outlier_mask', MM_outlier_mask)
plot.plot_mask_chip(htbam_experiment, mask_name='MM_outlier_mask')

# Apply the mask to the fits and save the results:
htbam_experiment.apply_mask(run_name='masked_fits_with_kcat',
                            dep_variables = ['v_max', 'K_m', 'r_squared', 'kcat'], 
                            save_as = 'masked_fits_with_kcat_filtered',
                            mask_names = ['MM_outlier_mask'])

# Also apply the mask to the slope data:
htbam_experiment.apply_mask(run_name='masked_V0_div_E_vs_conc',
                            dep_variables = ['V0_div_E'], 
                            save_as = 'masked_V0_div_E_vs_conc_filtered',
                            mask_names = ['MM_outlier_mask'])
                            

Transforming 1 objects of type [<class 'htbam_db_api.data.Data2D'>] with shapes [(1792, 4)].
Evaluating expression: ~(np.abs(a.chamber.K_m - np.nanmean(a.sample.K_m)) > z_score_threshold * np.nanstd(a.sample.K_m))
Transforming 1 objects of type [<class 'htbam_db_api.data.Data2D'>] with shapes [(1792, 4)].
Evaluating expression: ~(np.abs(a.chamber.kcat - np.nanmean(a.sample.kcat)) > z_score_threshold * np.nanstd(a.sample.kcat))
Transforming 1 objects of type [<class 'htbam_db_api.data.Data2D'>] with shapes [(1792, 1)].
Evaluating expression: ~(np.abs(a.chamber.concentration - np.nanmean(a.sample.concentration)) > z_score_threshold * np.nanstd(a.sample.concentration))
Transforming 3 objects of type [<class 'htbam_db_api.data.Data2D'>, <class 'htbam_db_api.data.Data2D'>, <class 'htbam_db_api.data.Data2D'>] with shapes [(1792, 1), (1792, 1), (1792, 1)].
Evaluating expression: a.mask.magnitude & b.mask.magnitude & c.mask.magnitude


In [ ]:
# # First, we make a boolean mask of all chambers that are within 3 standard deviations of the mean K_m across the entire device.
# sample_Km_mean = transform_data(
#     data_objs = [htbam_experiment.get_run('masked_fits_with_kcat_filtered')],
#     expr=f'~(np.abs(a.chamber.K_m - np.nanmean(a.sample.K_m)) > z_score_threshold * np.nanstd(a.sample.K_m))',
#     expression_vars={'z_score_threshold': z_score_threshold},
#     output_name='sample_Km_mean'
# )

# # add analysis

# htbam_experiment.set_run('sample_Km_mean', sample_Km_mean)

Transforming 1 objects of type [<class 'htbam_db_api.data.Data2D'>] with shapes [(1792, 4)].
Evaluating expression: ~(np.abs(a.chamber.K_m - np.nanmean(a.sample.K_m)) > z_score_threshold * np.nanstd(a.sample.K_m))


In [ ]:
# Expects 'V0_div_E'
plot.plot_MM_div_E_chip(experiment=htbam_experiment, analysis_name='masked_V0_div_E_vs_conc_filtered', 
                                model_fit_name='masked_fits_with_kcat_filtered',
                                dep_var_name='V0_div_E',
                                )

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/ma/core.py:2826: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/matplotlib/cbook.py:1398: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/ma/core.py:2358: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.



## 7. Export to CSV

In [123]:
htbam_experiment.export_run_data_raw(run_name='masked_fits_with_kcat_filtered')
htbam_experiment.export_run_data_processed(run_name='masked_fits_with_kcat_filtered')

Run data exported to /home/nfreitas/workspace/htbam_refactor_12_10_25/htbam_notebooks/htbam_notebooks/masked_fits_with_kcat_filtered_data.csv
Processed run data exported to /home/nfreitas/workspace/htbam_refactor_12_10_25/htbam_notebooks/htbam_notebooks/masked_fits_with_kcat_filtered_processed_data.csv


/home/nfreitas/workspace/htbam_refactor_12_10_25/htbam_analysis/src/htbam_analysis/analysis/experiment.py:187: RuntimeWarning:

Mean of empty slice

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1878: RuntimeWarning:

Degrees of freedom <= 0 for slice.



In [21]:
# This takes long!
# Saving the figure takes around 2-5 minutes...
htbam_experiment.export_mm_subplots(
                           analysis_name='kinetics_LMET_conc_fits_masked',
                           model_fit_name='kinetics_LMET_MM_fits_masked',
                           model_pred_data_name = 'kinetics_LMET_MM_pred_data',
                           export_path = 'mm_subplots.pdf')

  0%|          | 0/56 [00:00<?, ?it/s]/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/ma/core.py:2826: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/matplotlib/cbook.py:1398: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/ma/core.py:2358: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1583: RuntimeWarning:

All-NaN slice encountered

100%|██████████| 56/56 [00:12<00:00,  4.33it/s]


Exported MM subplots to mm_subplots.pdf


In [60]:
htbam_experiment.export_mm_subplots_by_sample(
                           analysis_name='masked_V0_div_E_vs_conc_filtered',
                           model_fit_name='masked_fits_with_kcat_filtered',
                           dep_var_label='V0_div_E',
                           export_path = 'mm_subplots4.pdf')

Plotting samples...


100%|██████████| 195/195 [00:01<00:00, 113.69it/s]


Exporting MM subplots by sample to mm_subplots4.pdf
Exported MM subplots by sample to mm_subplots4.pdf
